# Camera Calibration

In camera calibration, a chessboard (or checkerboard) pattern is commonly used as a reference object to estimate the camera's intrinsic and extrinsic parameters. The key reasons for using a chessboard in camera calibration are:

1. Feature Detection
The chessboard pattern consists of regularly spaced squares, which provides a strong contrast between the black and white squares. This allows for easy and accurate detection of corners (or intersections) in the pattern. These corners serve as feature points for calibration algorithms.

2. Known Geometry
The exact dimensions of the chessboard squares are known, providing a fixed real-world coordinate system. This allows the calibration process to map the 2D image points (from the captured images) to corresponding 3D world points (on the chessboard).

3. Multiple Views
By capturing images of the chessboard from different angles and distances, the camera's intrinsic parameters (such as focal length, optical center, and lens distortion) and extrinsic parameters (rotation and translation with respect to the chessboard) can be accurately estimated.

4. Distortion Correction
The chessboard's well-defined grid structure helps in identifying lens distortions (such as radial or tangential distortion) by comparing the expected grid lines with the distorted image. This allows for the correction of such distortions in the calibration process.

5. Robustness
The regularity and symmetry of the chessboard pattern ensure that it can be detected reliably even under varying lighting conditions and angles, making it a robust choice for calibration.

In summary, the chessboard pattern is used because it offers easily detectable and consistent reference points, facilitating the accurate calibration of a camera's lens and sensor system.

In [14]:
import cv2
import numpy as np

# === Parameters ===
chessboard_size = (9, 6)       # Number of inner corners (cols, rows)
square_size = 25               # Real-world square size in mm (or any unit)

max_frames = 20                # Number of valid frames to collect


In [15]:
# === Prepare object points like (0,0,0), (1,0,0), ..., (8,5,0)
objp = np.zeros((chessboard_size[0]*chessboard_size[1], 3), np.float32)
objp[:, :2] = np.mgrid[0:chessboard_size[0], 0:chessboard_size[1]].T.reshape(-1, 2)
objp *= square_size

objpoints = []  # 3D points in world
imgpoints = []  # 2D image points

cap = cv2.VideoCapture(0)  # Use default webcam

In [16]:
print("[INFO] Press SPACE to capture frame when corners are detected.")
print("[INFO] Press ESC to quit.")
while True:
    ret, frame = cap.read()
    if not ret:
        print("[ERROR] Failed to capture image.")
        break

    gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)

    # Detect corners
    found, corners = cv2.findChessboardCorners(gray, chessboard_size, None)

    if found:
        cv2.drawChessboardCorners(frame, chessboard_size, corners, found)
        cv2.putText(frame, f'Corners found - Press SPACE to save ({len(objpoints)}/{max_frames})',
                    (20, 30), cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0, 255, 0), 2)

    cv2.imshow("Calibration - Webcam", frame)
    key = cv2.waitKey(1) & 0xFF

    if key == 27:  # ESC
        print("[INFO] Exiting without full calibration.")
        break
    elif key == 32 and found:  # SPACE
        objpoints.append(objp.copy())
        imgpoints.append(corners)
        print(f"[INFO] Frame {len(objpoints)} captured.")
        if len(objpoints) >= max_frames:
            print("[INFO] Collected required frames.")
            break


[INFO] Press SPACE to capture frame when corners are detected.
[INFO] Press ESC to quit.
[INFO] Frame 1 captured.
[INFO] Frame 2 captured.
[INFO] Frame 3 captured.
[INFO] Frame 4 captured.
[INFO] Frame 5 captured.
[INFO] Frame 6 captured.
[INFO] Frame 7 captured.
[INFO] Frame 8 captured.
[INFO] Frame 9 captured.
[INFO] Frame 10 captured.
[INFO] Frame 11 captured.
[INFO] Frame 12 captured.
[INFO] Frame 13 captured.
[INFO] Frame 14 captured.
[INFO] Frame 15 captured.
[INFO] Frame 16 captured.
[INFO] Frame 17 captured.
[INFO] Frame 18 captured.
[INFO] Frame 19 captured.
[INFO] Frame 20 captured.
[INFO] Collected required frames.


In [17]:
cap.release()
cv2.destroyAllWindows()

if len(objpoints) >= 5:
    print("[INFO] Calibrating...")
    ret, camera_matrix, dist_coeffs, rvecs, tvecs = cv2.calibrateCamera(
        objpoints, imgpoints, gray.shape[::-1], None, None
    )

    print("\n=== Calibration Results ===")
    print("Camera Matrix:\n", camera_matrix)
    print("Distortion Coefficients:\n", dist_coeffs.ravel())

    # Save to file
    np.savez("webcam_calibration.npz",
             camera_matrix=camera_matrix,
             dist_coeffs=dist_coeffs,
             rvecs=rvecs,
             tvecs=tvecs)

    print("[INFO] Calibration data saved to webcam_calibration.npz")

else:
    print("[WARN] Not enough valid frames collected. Calibration aborted.")

[INFO] Calibrating...

=== Calibration Results ===
Camera Matrix:
 [[1.12493693e+03 0.00000000e+00 4.84238557e+02]
 [0.00000000e+00 1.13315799e+03 2.92572091e+02]
 [0.00000000e+00 0.00000000e+00 1.00000000e+00]]
Distortion Coefficients:
 [ -1.68771598   6.99790507  -0.02273382  -0.0895032  -16.9426976 ]
[INFO] Calibration data saved to webcam_calibration.npz


In [ ]:
aruco_id = 3
marker_size_pixels = 600
marker_size_mm = 50.0  # Set to the printed marker's actual side length in mm
output_file = f"aruco_id{aruco_id}.png"

if not 0 <= aruco_id < 50:
    raise ValueError("aruco_id must be between 0 and 49")

# === Generate and save the marker image ===
dictionary = cv2.aruco.getPredefinedDictionary(cv2.aruco.DICT_4X4_50)
marker = cv2.aruco.generateImageMarker(dictionary, aruco_id, marker_size_pixels)

if not cv2.imwrite(output_file, marker):
    raise OSError(f"Could not save marker to {output_file}")
print(f"Saved marker ID {aruco_id} to {output_file}")

# === Detect the marker live and estimate its pose ===
cap.release()  # The earlier calibration cell released the camera
cap = cv2.VideoCapture(0)
if not cap.isOpened():
    raise RuntimeError("Could not open the webcam")

parameters = cv2.aruco.DetectorParameters()
if hasattr(cv2.aruco, "ArucoDetector"):
    detector = cv2.aruco.ArucoDetector(dictionary, parameters)

# 3D corners of the marker in its own frame (order matches detector output)
half = marker_size_mm / 2
marker_obj_points = np.array([
    [-half,  half, 0],
    [ half,  half, 0],
    [ half, -half, 0],
    [-half, -half, 0],
], dtype=np.float32)

print("[INFO] Press ESC to quit.")
while True:
    ret, frame = cap.read()
    if not ret:
        print("[ERROR] Failed to capture image.")
        break

    if hasattr(cv2.aruco, "ArucoDetector"):
        marker_corners, ids, _ = detector.detectMarkers(frame)
    else:
        marker_corners, ids, _ = cv2.aruco.detectMarkers(
            frame, dictionary, parameters=parameters
        )

    if ids is not None:
        cv2.aruco.drawDetectedMarkers(frame, marker_corners, ids)

        for i, detected_id in enumerate(ids.flatten()):
            if detected_id == aruco_id:
                ok, rvec, tvec = cv2.solvePnP(
                    marker_obj_points, marker_corners[i].reshape(4, 2),
                    camera_matrix, dist_coeffs,
                    flags=cv2.SOLVEPNP_IPPE_SQUARE
                )
                if not ok:
                    continue
                cv2.drawFrameAxes(
                    frame, camera_matrix, dist_coeffs,
                    rvec, tvec, marker_size_mm * 0.5
                )
                x, y, z = tvec.flatten()
                cv2.putText(
                    frame, f"ID {aruco_id}: x={x:.1f}, y={y:.1f}, z={z:.1f} mm",
                    (10, 30), cv2.FONT_HERSHEY_SIMPLEX, 0.6,
                    (0, 255, 0), 2
                )

    cv2.imshow("ArUco detection", frame)
    if cv2.waitKey(1) & 0xFF == 27:
        break

cap.release()
cv2.destroyAllWindows()

### Parameters in the camera matrix and the distortion parameters can be used in distance estimation. 